# Tutorial 4: Conversation Management with Claude

This tutorial teaches you how to manage multi-turn conversations, context, and conversation state with Claude.

## What You'll Learn

- Building multi-turn conversations
- Managing conversation context and history
- Context window limits and token management
- Conversation persistence and resumption
- Advanced context techniques
- Memory and state management patterns

## Why Conversation Management Matters

**Pattern:** Context Management & Codebase Understanding

**Impact:** 30% improvement in response accuracy through strategic context management (source: patterns/context-management.md)

Good conversation management enables:
- Natural, coherent multi-turn dialogues
- Maintained context across interactions
- Efficient token usage
- Better user experience

---

## Setup

In [ ]:
!pip install anthropic python-dotenv

In [ ]:
import os
import json
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()

print("✓ Setup complete!")

## Example 1: Basic Multi-Turn Conversation

Let's start with a simple back-and-forth conversation:

In [ ]:
# Start a conversation
conversation = []

# Turn 1: User asks a question
conversation.append({
    "role": "user",
    "content": "What is a binary search tree?"
})

response1 = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=300,
    messages=conversation
)

assistant_response_1 = response1.content[0].text
conversation.append({
    "role": "assistant",
    "content": assistant_response_1
})

print("Turn 1:")
print(f"User: {conversation[0]['content']}")
print(f"\nAssistant: {assistant_response_1}")
print("\n" + "="*60 + "\n")

In [ ]:
# Turn 2: Follow-up question
conversation.append({
    "role": "user",
    "content": "Can you show me a Python implementation?"
})

response2 = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=500,
    messages=conversation
)

assistant_response_2 = response2.content[0].text
conversation.append({
    "role": "assistant",
    "content": assistant_response_2
})

print("Turn 2:")
print(f"User: {conversation[2]['content']}")
print(f"\nAssistant: {assistant_response_2}")

### Key Observation

Notice how Claude understood "Can you show me a Python implementation?" refers to the BST from the previous message. This is because we included the full conversation history.

## Example 2: Conversation Helper Class

Let's create a reusable conversation manager:

In [ ]:
class ConversationManager:
    """Manages a conversation with Claude."""
    
    def __init__(self, system_prompt=None, model="claude-3-5-sonnet-20241022"):
        self.messages = []
        self.system_prompt = system_prompt
        self.model = model
        self.total_input_tokens = 0
        self.total_output_tokens = 0
    
    def add_user_message(self, content):
        """Add a user message to the conversation."""
        self.messages.append({"role": "user", "content": content})
    
    def add_assistant_message(self, content):
        """Add an assistant message to the conversation."""
        self.messages.append({"role": "assistant", "content": content})
    
    def send_message(self, user_message, max_tokens=1024):
        """Send a message and get Claude's response."""
        self.add_user_message(user_message)
        
        kwargs = {
            "model": self.model,
            "max_tokens": max_tokens,
            "messages": self.messages
        }
        
        if self.system_prompt:
            kwargs["system"] = self.system_prompt
        
        response = client.messages.create(**kwargs)
        
        # Track token usage
        self.total_input_tokens += response.usage.input_tokens
        self.total_output_tokens += response.usage.output_tokens
        
        assistant_response = response.content[0].text
        self.add_assistant_message(assistant_response)
        
        return assistant_response
    
    def get_stats(self):
        """Get conversation statistics."""
        return {
            "turns": len(self.messages) // 2,
            "total_messages": len(self.messages),
            "input_tokens": self.total_input_tokens,
            "output_tokens": self.total_output_tokens,
            "total_tokens": self.total_input_tokens + self.total_output_tokens
        }
    
    def reset(self):
        """Reset the conversation."""
        self.messages = []
        self.total_input_tokens = 0
        self.total_output_tokens = 0

# Test the conversation manager
chat = ConversationManager(
    system_prompt="You are a helpful Python programming tutor. Provide concise, clear explanations."
)

response1 = chat.send_message("What is list comprehension in Python?")
print("Turn 1:")
print(response1)
print("\n" + "="*60 + "\n")

response2 = chat.send_message("Show me 3 examples")
print("Turn 2:")
print(response2)
print("\n" + "="*60 + "\n")

print("Conversation Stats:")
print(json.dumps(chat.get_stats(), indent=2))

## Example 3: Context Window Management

Claude has a maximum context window. Let's learn to manage it:

In [ ]:
class SmartConversationManager(ConversationManager):
    """Conversation manager with automatic context window management."""
    
    def __init__(self, system_prompt=None, model="claude-3-5-sonnet-20241022", max_context_tokens=100000):
        super().__init__(system_prompt, model)
        self.max_context_tokens = max_context_tokens
    
    def estimate_tokens(self, text):
        """Rough token estimation (4 chars ≈ 1 token)."""
        return len(text) // 4
    
    def get_context_size(self):
        """Estimate current context size."""
        total = 0
        for msg in self.messages:
            if isinstance(msg["content"], str):
                total += self.estimate_tokens(msg["content"])
        return total
    
    def trim_context(self, keep_recent=4):
        """Remove old messages, keeping the most recent ones."""
        if len(self.messages) > keep_recent:
            removed = len(self.messages) - keep_recent
            self.messages = self.messages[-keep_recent:]
            return removed
        return 0
    
    def send_message(self, user_message, max_tokens=1024, auto_trim=True):
        """Send message with automatic context management."""
        # Check context size
        estimated_size = self.get_context_size() + self.estimate_tokens(user_message)
        
        if auto_trim and estimated_size > self.max_context_tokens * 0.8:
            removed = self.trim_context(keep_recent=6)
            if removed > 0:
                print(f"⚠️  Context trimmed: removed {removed} old messages\n")
        
        return super().send_message(user_message, max_tokens)

# Test with context management
smart_chat = SmartConversationManager(
    max_context_tokens=500  # Very low for demonstration
)

# Have a long conversation
for i in range(5):
    response = smart_chat.send_message(f"Tell me fact #{i+1} about Python")
    print(f"\nTurn {i+1}: {response[:100]}...")
    print(f"Context size: ~{smart_chat.get_context_size()} tokens")
    print(f"Messages in memory: {len(smart_chat.messages)}")
    print("-" * 60)

## Example 4: Conversation Persistence

Save and load conversations:

In [ ]:
import json
from datetime import datetime

class PersistentConversation(ConversationManager):
    """Conversation manager with save/load capability."""
    
    def save(self, filename):
        """Save conversation to file."""
        data = {
            "timestamp": datetime.now().isoformat(),
            "model": self.model,
            "system_prompt": self.system_prompt,
            "messages": self.messages,
            "stats": self.get_stats()
        }
        
        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)
        
        print(f"✓ Conversation saved to {filename}")
    
    @classmethod
    def load(cls, filename):
        """Load conversation from file."""
        with open(filename, 'r') as f:
            data = json.load(f)
        
        conversation = cls(
            system_prompt=data.get("system_prompt"),
            model=data.get("model", "claude-3-5-sonnet-20241022")
        )
        conversation.messages = data.get("messages", [])
        
        print(f"✓ Conversation loaded from {filename}")
        print(f"  Timestamp: {data.get('timestamp')}")
        print(f"  Messages: {len(conversation.messages)}")
        
        return conversation

# Test persistence
persistent_chat = PersistentConversation(
    system_prompt="You are a helpful coding assistant."
)

persistent_chat.send_message("What is a closure in JavaScript?")
persistent_chat.send_message("Give me an example")

# Save
persistent_chat.save("/tmp/conversation.json")

# Load
loaded_chat = PersistentConversation.load("/tmp/conversation.json")

# Continue the conversation
response = loaded_chat.send_message("Can you explain the example?")
print("\nContinued conversation:")
print(response)

## Example 5: System Prompt Evolution

You can't change the system prompt mid-conversation, but you can work around it:

In [ ]:
# Approach 1: Include instructions in user messages
chat = ConversationManager(
    system_prompt="You are a helpful assistant."
)

# Start with general conversation
response1 = chat.send_message("Hello! What can you help me with?")
print("General mode:")
print(response1)
print("\n" + "="*60 + "\n")

# Switch to specific mode via user message
response2 = chat.send_message(
    """From now on, you are a Python code reviewer. 
    For any code I share, provide:
    1. Security issues
    2. Performance concerns
    3. Best practice violations
    
    Here's some code to review:
    
    ```python
    def process_data(data):
        result = []
        for item in data:
            result.append(item * 2)
        return result
    ```
    """
)

print("Code review mode:")
print(response2)

## Example 6: Conversation Summarization

For very long conversations, periodically summarize to save tokens:

In [ ]:
def summarize_conversation(messages):
    """Create a summary of the conversation."""
    
    # Create a summary request
    conversation_text = "\n\n".join([
        f"{msg['role'].upper()}: {msg['content']}"
        for msg in messages
    ])
    
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": f"""Summarize this conversation in 2-3 sentences, 
            capturing the key points and any decisions made:
            
            {conversation_text}
            
            Format: "This conversation covered..."
            """
        }]
    )
    
    return response.content[0].text

# Test summarization
test_conversation = [
    {"role": "user", "content": "What is Docker?"},
    {"role": "assistant", "content": "Docker is a containerization platform..."},
    {"role": "user", "content": "How is it different from a VM?"},
    {"role": "assistant", "content": "Containers are more lightweight than VMs..."}
]

summary = summarize_conversation(test_conversation)
print("Conversation Summary:")
print(summary)
print("\n" + "="*60 + "\n")

# Now you can replace old messages with the summary
compressed_conversation = [
    {
        "role": "user",
        "content": f"Previous conversation summary: {summary}"
    }
]

print("Original length:", len(str(test_conversation)))
print("Compressed length:", len(str(compressed_conversation)))
print(f"Saved: {100 - (len(str(compressed_conversation)) / len(str(test_conversation)) * 100):.1f}%")

## Example 7: Multi-Topic Conversation Tracking

Track different topics within a single conversation:

In [ ]:
class TopicAwareConversation(ConversationManager):
    """Conversation manager that tracks topics."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.topics = []
    
    def detect_topic_change(self, user_message):
        """Use Claude to detect if the topic changed."""
        if len(self.messages) < 2:
            return None
        
        recent_context = self.messages[-4:] if len(self.messages) >= 4 else self.messages
        
        response = client.messages.create(
            model="claude-3-5-haiku-20241022",  # Use faster model
            max_tokens=50,
            messages=recent_context + [{
                "role": "user",
                "content": f"""Is this new message about a different topic than the previous conversation?
                
                New message: "{user_message}"
                
                Respond with ONLY 'yes' or 'no'."""
            }]
        )
        
        return "yes" in response.content[0].text.lower()
    
    def send_message(self, user_message, max_tokens=1024, track_topic=True):
        """Send message with topic tracking."""
        
        if track_topic and len(self.messages) > 0:
            changed = self.detect_topic_change(user_message)
            if changed:
                self.topics.append({
                    "message_index": len(self.messages),
                    "topic_preview": user_message[:50]
                })
                print(f"📌 New topic detected: {user_message[:50]}...\n")
        
        return super().send_message(user_message, max_tokens)

# Test topic tracking
topic_chat = TopicAwareConversation()

topic_chat.send_message("Explain how Python decorators work")
topic_chat.send_message("Can you show an example?")
topic_chat.send_message("Now tell me about JavaScript promises")
topic_chat.send_message("What's the difference with async/await?")

print("\n" + "="*60)
print(f"Topics discussed: {len(topic_chat.topics)}")
for i, topic in enumerate(topic_chat.topics):
    print(f"{i+1}. {topic['topic_preview']}...")

## Example 8: Contextual Memory with Document Grounding

**Pattern:** Persistent Memory (Claude.md)

Include relevant documentation in each request:

In [ ]:
class DocumentGroundedConversation(ConversationManager):
    """Conversation with persistent document context."""
    
    def __init__(self, documents=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.documents = documents or {}
    
    def add_document(self, name, content):
        """Add a reference document."""
        self.documents[name] = content
    
    def send_message(self, user_message, max_tokens=1024, include_docs=True):
        """Send message with document context."""
        
        if include_docs and self.documents:
            # Prepend documents to the message
            doc_context = "\n\n".join([
                f"<document name=\"{name}\">\n{content}\n</document>"
                for name, content in self.documents.items()
            ])
            
            enhanced_message = f"{doc_context}\n\n{user_message}"
        else:
            enhanced_message = user_message
        
        return super().send_message(enhanced_message, max_tokens)

# Test with documentation
grounded_chat = DocumentGroundedConversation()

# Add project context
grounded_chat.add_document(
    "project_readme",
    """# My Project
    Tech Stack: Python 3.11, FastAPI, PostgreSQL
    Architecture: Microservices with REST APIs
    Testing: pytest, 80% coverage minimum
    """
)

grounded_chat.add_document(
    "coding_standards",
    """# Coding Standards
    - Use type hints
    - Max line length: 100
    - Use black for formatting
    - All functions must have docstrings
    """
)

response = grounded_chat.send_message(
    "I need to write a new API endpoint for user registration. What should I keep in mind?",
    include_docs=True
)

print("Response with document context:")
print(response)

## Interactive Exercise: Build a Chatbot

Create a simple interactive chatbot:

In [ ]:
def interactive_chat(system_prompt="You are a helpful assistant.", max_turns=10):
    """Simple interactive chat interface."""
    
    chat = ConversationManager(system_prompt=system_prompt)
    
    print("Chat started! Type 'quit' to exit, 'stats' for statistics.")
    print("="*60)
    
    for turn in range(max_turns):
        # In a real notebook, you'd use input()
        # For demo, we'll use predefined messages
        demo_messages = [
            "Hello! How are you?",
            "What's the weather like?",
            "stats"
        ]
        
        if turn >= len(demo_messages):
            break
            
        user_input = demo_messages[turn]
        
        print(f"\nYou: {user_input}")
        
        if user_input.lower() == 'quit':
            print("Goodbye!")
            break
        
        if user_input.lower() == 'stats':
            print(json.dumps(chat.get_stats(), indent=2))
            continue
        
        response = chat.send_message(user_input)
        print(f"\nAssistant: {response}")
        print("-" * 60)
    
    return chat

# Run demo
chat_session = interactive_chat(
    system_prompt="You are a friendly chatbot that gives very brief responses."
)

## Production Best Practices

### 1. Token Budget Management

```python
def estimate_cost(input_tokens, output_tokens, model="claude-3-5-sonnet-20241022"):
    """Estimate cost of API call."""
    pricing = {
        "claude-3-5-sonnet-20241022": {"input": 3.0, "output": 15.0},
        "claude-3-5-haiku-20241022": {"input": 0.80, "output": 4.0},
    }
    
    rates = pricing.get(model, pricing["claude-3-5-sonnet-20241022"])
    
    cost = (
        (input_tokens / 1_000_000) * rates["input"] +
        (output_tokens / 1_000_000) * rates["output"]
    )
    
    return cost
```

### 2. Prompt Caching

**Impact:** 90% cost savings on repeated context (source: patterns/optimization/token-efficiency.md)

```python
# Use prompt caching for repeated content
response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    system=[
        {
            "type": "text",
            "text": "Large system prompt or documentation...",
            "cache_control": {"type": "ephemeral"}
        }
    ],
    messages=messages
)
```

### 3. Conversation Expiry

```python
from datetime import datetime, timedelta

class ExpiringConversation(ConversationManager):
    def __init__(self, expiry_minutes=30, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.last_activity = datetime.now()
        self.expiry_minutes = expiry_minutes
    
    def is_expired(self):
        return datetime.now() - self.last_activity > timedelta(minutes=self.expiry_minutes)
    
    def send_message(self, user_message, max_tokens=1024):
        if self.is_expired():
            self.reset()
            print("⚠️  Conversation expired and reset")
        
        self.last_activity = datetime.now()
        return super().send_message(user_message, max_tokens)
```

### 4. Conversation Branching

```python
def create_branch(conversation, from_turn):
    """Create a conversation branch from a specific turn."""
    branched = ConversationManager(
        system_prompt=conversation.system_prompt,
        model=conversation.model
    )
    branched.messages = conversation.messages[:from_turn].copy()
    return branched
```

## Key Takeaways

1. **Message History is Key**: Always include full conversation context for coherent multi-turn dialogues

2. **Manage Context Size**: Monitor token usage and trim old messages when needed

3. **Save State**: Persist conversations for resumption across sessions

4. **System Prompts**: Set behavior at conversation start; use user messages to adjust mid-conversation

5. **Summarization**: Compress long conversations to save tokens while preserving context

6. **Document Grounding**: Include persistent context (docs, standards) for consistent responses

7. **Prompt Caching**: Use caching for 90% cost savings on repeated content

## Common Patterns

- **Customer Support**: Multi-turn help with context persistence
- **Code Review**: Maintain project standards across conversation
- **Tutoring**: Track learning progress and adapt
- **Data Analysis**: Iterative exploration with state

## Next Steps

- **Tutorial 5**: Learn production patterns for scaling Claude applications

## Resources

- [Context Management Pattern](../patterns/context-management.md)
- [Claude.md Persistent Memory](../patterns/claude-md-persistent-memory.md)
- [Token Efficiency Guide](../patterns/optimization/token-efficiency.md)
- [Anthropic Messages API](https://docs.anthropic.com/en/api/messages)